In [6]:
from sklearn import svm
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

X, y = load_digits(return_X_y=True)

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

sv_classifiers = {
    "LinearSVC"    : svm.LinearSVC(),
    "SVC(linear)"  : svm.SVC(kernel='linear'),
    "SVC(poly=1)"  : svm.SVC(kernel='poly', degree=1),
    "SVC(poly=2)"  : svm.SVC(kernel='poly', degree=2),
    "SVC(poly=3)"  : svm.SVC(kernel='poly', degree=3),
    "SVC(rbf,.5)"  : svm.SVC(kernel='rbf', gamma=0.5),
    "SVC(rbf,1.0)" : svm.SVC(kernel='rbf', gamma=1.0),
    "SVC(rbf,2.0)" : svm.SVC(kernel='rbf', gamma=2.0)}

for name, clf in sv_classifiers.items():
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", clf)])
    pipe.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, pipe.predict(X_te))
    print(f"{name:12s} | accuracy = {acc:.4f}")


LinearSVC    | accuracy = 0.9556
SVC(linear)  | accuracy = 0.9750
SVC(poly=1)  | accuracy = 0.9861
SVC(poly=2)  | accuracy = 0.9778
SVC(poly=3)  | accuracy = 0.9722
SVC(rbf,.5)  | accuracy = 0.3861
SVC(rbf,1.0) | accuracy = 0.1361
SVC(rbf,2.0) | accuracy = 0.1167


In [4]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

X, y = load_digits(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf"))])

param_grid = {
    "clf__C": [0.1, 1, 10, 100],
    "clf__gamma": ["scale", "auto", 0.01, 0.1, 1, 10]}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1)

grid.fit(X_tr, y_tr)
y_pred = grid.best_estimator_.predict(X_te)
print(f"{accuracy_score(y_te, y_pred):.4f}")
print(grid.best_params_)


0.9750
{'clf__C': 1, 'clf__gamma': 'auto'}


In [7]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.metrics import r2_score

X, y = load_diabetes(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR())])

# Grid s tri kernela, svaki s po 2 hiperparametra
param_grid = [
    # linear: C i epsilon
    {"svr__kernel": ["linear"],
     "svr__C": [0.1, 10],
     "svr__epsilon": [0.1, 0.2]},
    # poly: C i degree
    {"svr__kernel": ["poly"],
     "svr__C": [1, 10],
     "svr__degree": [2, 3]},
    # rbf: C i gamma
    {"svr__kernel": ["rbf"],
     "svr__C": [1, 100],
     "svr__gamma": ["scale", 0.1]}]

cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="r2",
    cv=cv,
    n_jobs=-1,
    refit=True,)

grid.fit(X_tr, y_tr)
y_pred = grid.best_estimator_.predict(X_te)
print(f"{r2_score(y_te, y_pred):.4f}")


0.4302
